In [ ]:
#--- Differentiable Graph Optimization -- Train a model to assign useful scores to relationships, then use those scores to 
# prioritize which relationships deserve attention.
"""
Suppose a research system has three candidate relationships:
- Biomarker result → Treatment
- Treatment → Documented response
- Diagnosis → Hospital location
A researcher has labeled the first two as more relevant to a particular research task. Can a model learn to assign them 
higher scores?

The scores are trainable parameters. A loss function measures how far the predictions are from the labels, and gradient 
descent updates the scores.

This example learns three independent relevance scores. It does not learn a full graph structure or solve a complex 
graph optimization problem.

        Block 1 — Define relationships
        Three candidate edges in the oncology graph

        Block 2 — Assign initial scores
        Start with three scores, all equal to zero

        Block 3 — Compare with labels
        Relevant = 1; not relevant = 0

        Block 4 — Calculate the error
        Measure how far the scores are from the labelled answers

        Block 5 — Update the scores
        Use gradient descent to reduce the error

        Block 6 — Rank relationships
        Display the highest-scoring relationships first
"""
import torch
from torch import nn
#---------Block 1 — Define the candidate relationships
edges = [                            
    "Biomarker_to_Treatment",        # Which treatment was given in the context of a biomarker result?
    "Treatment_to_Response",         # What outcome was documented after treatment?
    "Diagnosis_to_Location"          # At which hospital or facility was the diagnosis recorded?
]
#------End of Block 1 — Define the candidate relationships

#----Block 3 — Understand the labels and probabilities
labels = torch.tensor([1.0, 1.0, 0.0])
# Above 1 means relevant and 0 means not relevant 
#----End of Block 3 — Understand the labels and probabilities 

#------------Block 2 — Create the trainable scores
edge_scores = nn.Parameter(torch.zeros(3))       
"""
torch.zeros(3) creates three numbers, initially all zero so none is preferred at the start. so Biomarker → Treatment = 0 , Treatment → Response = 0 , Diagnosis → Location = 0 
nn.Parameter(...) tells PyTorch these numbers are trainable parameters.
The optimizer can adjust these numbers during training.
"""
#---------End of Block 2 — Create the trainable scores

optimizer = torch.optim.Adam([edge_scores], lr=0.1)
loss_function = nn.BCEWithLogitsLoss()                 #----Block 4 — Measure the error using a loss function
"""
BCEWithLogitsLoss stands for Binary Cross-Entropy with Logits Loss.
In simple terms, it measures how poorly the model's raw scores match the binary labels.
- For a label of 1, the model should move its score upward so its sigmoid output approaches 1.
- For a label of 0, the model should move its score downward so its sigmoid output approaches 0.
"""
#--------Block 5 — Learn by updating the scores and Rank the relationships
for epoch in range(300):
    probabilities = torch.sigmoid(edge_scores)         # can convert each score into a value between 0 and 1: The sigmoid function transforms a logit into a probability-like value.
    loss = loss_function(edge_scores, labels)          #----Block 4 — Measure the error using a loss function
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
probabilities = torch.sigmoid(edge_scores).detach()
ranked = sorted(
    zip(edges, probabilities.tolist()),
    key=lambda item: item[1],
    reverse=True
)
"""
    Converts the learned scores into probability-like values.
    Pairs each value with its relationship.
    Sorts the relationships from highest score to lowest.
"""
#---------End of Block 5 — Learn by updating the scores and Rank the relationships
print("Learned relationship priorities:")
for edge, score in ranked:
    print(f"{edge}: {score:.3f}")

Learned relationship priorities:
Biomarker_to_Treatment: 0.996
Treatment_to_Response: 0.996
Diagnosis_to_Location: 0.004
